In [ ]:
! python --version

Python 3.10.12


# init
NOTE:
- This was tested with python version 3.12.2 on Mac Sonoma 14.1. Needed to import a downgraded version of numpy to support downgraded version of wandb.
- This was tested with python version 3.10.11 on Windows 11.

In [ ]:
! pip install transformers==4.36.2
! pip install wandb==0.17.1
! pip install accelerate==0.31.0
! pip install rouge_score==0.1.2
! pip install sacrebleu==2.4.2
! pip install sentencepiece==0.1.99
! pip install evaluate==0.4.3
! pip install nltk==3.8.1
! pip install jiwer==3.0.4
! pip install numpy==1.26.4

In [ ]:
from google.colab import drive
import sys
drive.mount('/content/drive')
sys.path.append('./drive/MyDrive/Colab Notebooks/vr-txt-corr/')

import numpy as np
import pandas as pd
import torch
import correction_palette as palette
from transformers import T5TokenizerFast, T5ForConditionalGeneration, Seq2SeqTrainingArguments
from correction_palette import *
from cursor_t5_modeling import CursorT5ForConditionalGeneration
import evaluate
import os
import copy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/deepspeed.py:23: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  war

Using CUDA


In [ ]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    gpu_id = 0

    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}")
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    device = torch.device(gpu_id)
    print(f"Using CUDA with {torch.cuda.get_device_name(i)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

Number of GPUs available: 1
GPU 0: NVIDIA L4
Using CUDA with NVIDIA L4


# Configure stuffs

In [ ]:
save_dir = './drive/MyDrive/Colab Notebooks/vr-txt-corr/'  # save directory for pretty much anything that is saved by this program (models, tokenizers, logs, etc.)
models_dir = f'{save_dir}saved_models/'
data_dir = f'{save_dir}corr_datasets_experimental/DELETION-INSERTION-MULTIPLE-REPLACEMENT-SINGLE/'

base_model_name = 'google/flan-t5-base'

NUM_SAVES = 2

PORTION = 1 # proportion ofdatasets to use (note: applies to each split)

random_seed = 42
palette.MAX_SOURCE_TOKENS = 512
palette.MAX_NEW_TOKENS = 512
palette.CURSOR_TOKEN = '<|>'
palette.TEXT_SEP_TOKEN = '<||>'
custom_special_tokens_dict = {'additional_special_tokens': [palette.CURSOR_TOKEN, palette.TEXT_SEP_TOKEN]}

gen_kwargs_override = {'num_beams': 24, 'max_new_tokens': palette.MAX_NEW_TOKENS, 'early_stopping': True, 'num_return_sequences': 24}  # these values might be changed in 'Sweep Configuration'

# Computing Metrics

In [ ]:
import difflib
import re


def prefix_key(k: str, prefix: str, sep: str) -> str:
    return f'{prefix}{sep}{k}'

def get_unique_words(text):
    #words = set(text.replace('_', ' ').replace(',', ' ').replace(':', ' ').replace(';', ' ').replace('.', ' ').replace('?', ' ').replace('(', ' ').split().split())
    list_words = re.split(',|\ |_|-|!|\+|\.|\\|\*|\?|\[|\^|\]|\$|\(|\)|\{|\}|\=|\||\:', text)
    words = set(list_words)
    if '' in words:
        words.remove('')
    return words

def get_split_index(split_list,index):
    start_index = 0
    end_index = 0
    for i in range(0,index+1):
        if i == 0:
            start_index = 0
            end_index = len(split_list[i])
        else:
            start_index+=len(split_list[i-1])+1
            end_index= start_index+len(split_list[i])
    return start_index,end_index

def get_diff_start_end(original,new):

    original = ' '+original
    new = ' ' +new
    s1 = original.split(' ')
    s2 = new.split(' ')
    matcher = difflib.SequenceMatcher(a=s1, b=s2)
    #print("Matching Sequences:")
    '''
    a_last_end = 0
    count = 0
    for match in matcher.get_matching_blocks():
        #print("Match             : {}".format(match))
        #print("Matching Sequence : {}".format(s1[match.a:match.a+match.size]))
        #print(match.a,match.a+match.size)
        if count>0:
            if match.a-a_last_end>0:
                #print("missmatch",a_last_end,match.a)
                #print("Miss_matching Sequence : {}".format(s1[a_last_end:match.a]))
                start,e = get_split_index(s1,a_last_end)
                s,end = get_split_index(s1,match.a-1)
                print("mismatch start end:",start,end)
                print("***miss match segment:", string1[start:end])


        count+=1
        a_last_end=match.a+match.size
    '''
    b_last_end = 0
    count = 0
    start_list = []
    end_list = []
    for match in matcher.get_matching_blocks():
        #print("Match             : {}".format(match))
        #print("Matching Sequence : {}".format(s2[match.b:match.b+match.size]))
        #print(match.b,match.b+match.size)
        if count>0:
            if match.b-b_last_end>0:
                #print("missmatch",b_last_end,match.b)
                #print("Miss_matching Sequence : {}".format(s2[b_last_end:match.b]))
                start,_ = get_split_index(s2,b_last_end)
                _,end = get_split_index(s2,match.b-1)
                start_list.append(start-1)
                end_list.append(end-1)

                #print("mismatch start end:",start,end)
                #print("***miss match segment:", new[start:end])
        count+=1
        b_last_end=match.b+match.size

    return start_list, end_list

def filter_corrections(response, sentence_without_marker,data):
    filtered_corrections = []
    filtered_orig_idx = []
    diff_start_list = []
    diff_end_list = []
    for filter_idx in range(len(response['corrections'])):
        origitem = response['corrections'][filter_idx]

        item = origitem.strip(" ")

        if item == sentence_without_marker:
            continue
        if item in filtered_corrections:
            continue

        item_words = get_unique_words(item.lower());
        original_and_input_words = get_unique_words(sentence_without_marker.lower()+" "+data['part'].lower())
        #print("item_words",item_words)
        #print("original_and_input_words",original_and_input_words)
        if not item_words.issubset(original_and_input_words):
            continue

        if data['part'].lower() not in item.lower():
            continue

        # and (response['correction probabilities'][filter_idx]>0):
        if len((data['part']+" "+sentence_without_marker).split()) < len(item.split()):
            continue

        filtered_corrections.append(origitem)
        filtered_orig_idx.append(filter_idx)

        (start_list, end_list) = get_diff_start_end(sentence_without_marker,item)
        diff_start_list.append(start_list)
        diff_end_list.append(end_list)
        '''
        if (response['correction probabilities'][filter_idx]>0.7):
            break
        else:
            print("prob <0.7")
            print(response['correction probabilities'][filter_idx], item)
        '''

    return filtered_corrections, filtered_orig_idx, diff_start_list, diff_end_list

def F1_score(ground_truth, result_list):
    num_of_results = float(len(result_list))
    recall = 0
    precision = 0
    if ground_truth in result_list:
        recall = 1
        precision = 1.0/num_of_results;
    else:
        recall = 0
        precision = 0

    F1 = 2*recall*precision/(recall+precision+0.00000000001)
    return F1,recall,precision

f1,recall,precision = F1_score('how are you',[])
print('recall',recall)
print("precision",precision)
print("F1",f1)


string1 ="It is not warm. It is summer."
string2 ="It is very warm. It is winter."
get_diff_start_end(string1,string2)

recall 0
precision 0
F1 0.0


([6, 23], [10, 30])

In [ ]:
def filter_invalid_samples(_label_str, _input_strs, _beam_str, _best_beam_str):
    label_str = []
    input_strs = []
    best_beam_str = []
    beam_str = []
    for i in range(len(_label_str)):
        e_sentence, phrase, edits, edits_made = _input_strs[i]
        e_sentence, phrase = tokenizer.decode(tokenizer(e_sentence)['input_ids'], skip_special_tokens=True), tokenizer.decode(tokenizer(phrase)['input_ids'], skip_special_tokens=True)
        label = _label_str[i]
        if label == e_sentence:
            continue

        beam_group = _beam_str[i]
        label = label

        d_response = {'corrections': beam_group}
        d_data = {'part': phrase}

        pred_slice_before = beam_group[:5]
        num_correct_before = label in pred_slice_before
        filter_output = filter_corrections(d_response, e_sentence, d_data)

        pred_slice_after = filter_output[0][:5]
        num_correct_after = label in pred_slice_after
        if num_correct_after < num_correct_before:
            print(num_correct_before)
            print(num_correct_after)
            print(f'skipped (sent: |{e_sentence}|, phra: |{phrase}|)')
            print(f"cbefo: |{pred_slice_before[0]}|")
            print(f"cafte: |{pred_slice_after}|")
            print(f"label: |{label}|")
            continue

        label_str.append(label)
        input_strs.append((e_sentence, phrase, edits, edits_made))
        best_beam_str.append(_best_beam_str[i])
        beam_str.append(_beam_str[i])
    return label_str, input_strs, best_beam_str, beam_str

# filter beams based on criterion. save filtered_ids so that we can simulate slicing the top n beams and then filtering without recomputing/refiltering.
def filter_beams(label_str, input_strs, beam_str, best_beam_str):
    filtered_ids = []

    #print(f'beam_str: {np.array(beam_str).shape}')
    #print(f'label_str: {np.array(label_str).shape}')
    #print(f'input_strs: {np.array(input_strs).shape}')
    #print(f'input_strs: {input_strs}')
    for i in range(len(beam_str)): # filter invalid beams
        beam_group = beam_str[i]
        label = label_str[i]
        sentence, phrase, _, _= input_strs[i]

        d_response = {'corrections': beam_group}
        d_data = {'part': phrase}

        pred_slice_before = beam_group[:5]
        num_correct_before = label in pred_slice_before
        filter_output = filter_corrections(d_response, sentence, d_data)
        beam_str[i] = filter_output[0]
        if len(beam_str[i]) > 0:
            best_beam_str[i] = beam_str[i][0]
        filtered_ids.append(filter_output[1])
        pred_slice_after = beam_str[i][:5]
        num_correct_after = label in pred_slice_after
        if num_correct_after < num_correct_before:
            print(f'impossible')
    return filtered_ids, beam_str, best_beam_str

# computing f1, precision, and recall
def compute_fpr(_beam_str, label_str, filtered_ids, prefix: str = 'n_samp'):
    beam_str = copy.deepcopy(_beam_str)
    n_metrics = dict()
    n_table = []
    for j in range(24, 0, -1): # iterate over possible num_samples (to obtain n_metrics)
        f1_sum = 0
        recall_sum = 0
        precision_sum = 0

        for i in range(len(beam_str)):
            if filtered_ids is not None:
                while len(beam_str[i]) > 0 and filtered_ids[i][len(beam_str[i]) - 1] >= j:
                    # remove from end based on filtered_ids because filtered_ids represents the original indices of the beams.
                    # this way, we don't have to keep re-slicing the top n beams and re-filtering
                    beam_str[i].pop()

            metric_output = F1_score(label_str[i], beam_str[i])
            f1_sum += metric_output[0]
            recall_sum += metric_output[1]
            precision_sum += metric_output[2]
            if filtered_ids is None:
                beam_str[i].pop()

        f1 = f1_sum / len(beam_str)
        recall = recall_sum / len(beam_str)
        precision = precision_sum / len(beam_str)

        n_table.append({'n': j, 'f1': f1, 'recall': recall, 'precision': precision})

        n_metrics[f'n{j}/f1'] = f1
        n_metrics[f'n{j}/recall'] = recall
        n_metrics[f'n{j}/precision'] = precision

    n_metrics = {prefix_key(k, prefix,'/'): v for k, v in n_metrics.items()}

    # display tables
    n_table = pd.DataFrame(n_table).iloc[::-1].reset_index(drop=True)
    return n_metrics, n_table

In [ ]:
import os
import pandas as pd

def log_mismatches(input_texts, predictions, labels):
    """
    Logs mismatched cases where predictions do not match labels.

    Parameters:
    - input_texts (List[str]): The input strings.
    - predictions (List[str]): The predicted strings.
    - labels (List[str]): The label strings.
    """
    global mismatch_file
    mismatches = []

    for input_text, pred, label in zip(input_texts, predictions, labels):
        if pred.strip() != label.strip():
            mismatches.append({
                'input': input_text,
                'prediction': pred,
                'label': label
            })

    if mismatches:
        df = pd.DataFrame(mismatches)
        # Check if the file exists to determine whether to write headers
        if not os.path.isfile(mismatch_file):
            df.to_csv(mismatch_file, index=False)
        else:
            df.to_csv(mismatch_file, mode='a', header=False, index=False)
        print(f"Logged {len(mismatches)} mismatched cases to {mismatch_file}")
    else:
        print("No mismatches found in this batch.")

In [ ]:
topk =[1,3,5]

def flatten_stats(stats, topk):
    """
    Reorganize stats into a more compact DataFrame structure with columns:
    - category (edit_count, extra_word_count, etc.)
    - subcat (numerical value)
    - metric (top1, top3, top5)
    - value (accuracy rate)
    - count (number of matches)
    - total (total samples)
    """
    rows = []
    for category in stats:
        for subcat, data in stats[category].items():
            matches = data['matches']
            total = data['total']
            for i, k in enumerate(topk):
                rate = matches[i]/total if total != 0 else 0
                rows.append({
                    'category': category,
                    'subcat': subcat,
                    'metric': f'top{k}',
                    'accuracy': rate,
                    'matches': int(matches[i]),
                    'total': total
                })
    return pd.DataFrame(rows)

from IPython.display import display, HTML
def save_stats_to_csv(stats_df, model_path, triplet_comb):
    """
    Save statistics DataFrame to a CSV file.

    Args:
        stats_df (pd.DataFrame): DataFrame containing the statistics
        model_path (str): Path to save the model
        triplet_comb (str): Triplet combination identifier
    """
    try:
        stats_file = f'{model_path}/stats_triplets-{triplet_comb}-relax10.csv'
        os.makedirs(os.path.dirname(stats_file), exist_ok=True)
        stats_df.to_csv(stats_file, index=False)
        print(f"Saved detailed statistics to {stats_file}")

    except Exception as e:
        print(f"Warning: Failed to save statistics: {e}")

def display_stats_pivot(stats_df):
    """
    Display statistics in a pivot table format.
    Works in both notebook and standard Python environments.

    Args:
        stats_df (pd.DataFrame): DataFrame containing the statistics
    """
    try:
        # Create pivot table
        pivot_df = stats_df.pivot_table(
            index=['category', 'subcat'],
            columns='metric',
            values=['accuracy', 'matches', 'total'],
            aggfunc='first'
        ).round(4)

        # Try notebook display first, fall back to regular print
        try:
            from IPython.display import display
            display(pivot_df)
        except ImportError:
            print("\nStatistics Summary:")
            print("=" * 80)
            print(pivot_df.to_string())
            print("=" * 80)

    except Exception as e:
        print(f"Warning: Failed to display statistics: {e}")

def display_current_stats(stats_df: pd.DataFrame, processed_count: int):
    """
    Display current statistics using display_stats_pivot function,
    with added sample count information

    Args:
        stats_df (pd.DataFrame): Pre-flattened statistics DataFrame
        processed_count (int): Number of samples processed
    """
    print(f"\nStatistics after processing {processed_count} samples:")
    print("=" * 80)

    # Display stats using existing pivot function
    display_stats_pivot(stats_df)

    print("=" * 80)

def compute_stats(input_info, predictions, labels):

    stats = {
        'edit_count': defaultdict(lambda: {'matches': np.zeros(len(topk)), 'total': 0}),
        'extra_word_count': defaultdict(lambda: {'matches': np.zeros(len(topk)), 'total': 0}),
        'extra_word_added': defaultdict(lambda: {'matches': np.zeros(len(topk)), 'total': 0}),
        'part_word_count': defaultdict(lambda: {'matches': np.zeros(len(topk)), 'total': 0})
    }
    i=0
    for preds, label_str, (_, part, edits, edits_made) in zip(predictions, labels,
                                                    input_info):
        words_in_part = part.split()
        part_word_count = len(words_in_part)
        edits = ast.literal_eval(edits)
        edit_count = len(edits)
        edits_made = ast.literal_eval(edits_made)
        # Sort edits_made based on start position
        edits_made = sorted(edits_made, key=lambda x: x[1][0])
        words_edited = sum(em[1][1]-em[1][0] for em in edits_made)
        min_words_needed = edits_made[-1][1][1]-edits_made[0][1][0]
        if min_words_needed < 0:
            print(f"***min_words_needed < 0: {min_words_needed}, part: {part}, label: {label_str},edits_made: {edits_made}")
        extra_word_count = part_word_count - words_edited
        extra_word_added = part_word_count - min_words_needed
        if extra_word_added < 0:
            print(f"***extra_word_added < 0: {extra_word_added}, part: {part}, label: {label_str}, edits_made: {edits_made}")

        stats['edit_count'][edit_count]['total'] += 1
        stats['extra_word_count'][extra_word_count]['total'] += 1
        stats['extra_word_added'][extra_word_added]['total'] += 1
        stats['part_word_count'][part_word_count]['total'] += 1
        is_match_topk = 0
        for j in range(len(topk)):
            is_match_topk = label_str.strip() in preds[:topk[j]]
            stats['edit_count'][edit_count]['matches'][j] += is_match_topk
            stats['extra_word_count'][extra_word_count]['matches'][j] += is_match_topk
            stats['extra_word_added'][extra_word_added]['matches'][j] += is_match_topk
            stats['part_word_count'][part_word_count]['matches'][j] += is_match_topk

        i +=1
        if i % 500 == 0:
            # Flatten stats before displaying
            current_stats_df = flatten_stats(stats, topk)
            display_current_stats(current_stats_df, i)

    return stats

In [ ]:
from IPython.display import display, HTML

cer = evaluate.load('cer')
tkacc = TopKAccuracy()

def compute_metrics(gen_pred: GenPrediction):
    global triplet_comb
    label_ids = gen_pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_ids = gen_pred.predictions
    pred_ids[pred_ids == -100] = tokenizer.pad_token_id
    _input_strs = gen_pred.inputs_strs
    _label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    for lstr in _label_str:
        lstr = ' '.join([word for word in lstr.split(' ') if word != '|>' and word != '|>.'])

    _pred_str = tokenizer.batch_decode(np.array(pred_ids).reshape(-1, len(pred_ids[0][0])), skip_special_tokens=True)
    _pred_str = np.array(_pred_str, dtype=str)


    _best_beam_str = _pred_str[::gen_kwargs_override['num_return_sequences']].tolist()
    _beam_str = _pred_str.reshape(-1, gen_kwargs_override['num_return_sequences']).tolist()


    label_str, input_strs, best_beam_str, beam_str = filter_invalid_samples(_label_str, _input_strs, _beam_str, _best_beam_str)

    # compute metrics before filtering
    tkacc_metrics_unfiltered = tkacc.compute(predictions=beam_str, references=label_str, k_metrics=[1, 3, 5], strict_k=False)
    cer_metric_unfiltered = {'cer_unfiltered': cer.compute(predictions=best_beam_str, references=label_str)}


    # filter bad results based on filtering criterion. also save filtered_ids to avoid unecessary recomputations.
    filtered_ids, beam_str, best_beam_str = filter_beams(label_str, input_strs, beam_str, best_beam_str)

    stats = compute_stats(input_strs, beam_str, label_str)

    # Create the stats DataFrame
    stats_df = flatten_stats(stats, topk)

    # Save and display stats using the new functions
    save_stats_to_csv(stats_df, model_path, triplet_comb)
    print("\nFinal Statistics:")
    display_current_stats(stats_df, len(test_dataset))

    # compute metrics after filtering
    tkacc_metrics_filtered = tkacc.compute(predictions=beam_str, references=label_str, k_metrics=[1, 3, 5], strict_k=False)
    cer_metric_filtered = {'cer_filtered': cer.compute(predictions=best_beam_str, references=label_str)}

    # add prefixes
    tkacc_metrics_unfiltered = {prefix_key(k, 'tkacc_unfiltered', '/'): v for k, v in tkacc_metrics_unfiltered.items()}
    tkacc_metrics_filtered = {prefix_key(k, 'tkacc_filtered', '/'): v for k, v in tkacc_metrics_filtered.items()}


    # construct a table for non-FPR metrics
    metric_table = pd.DataFrame([{
        **cer_metric_unfiltered,
        **cer_metric_filtered,

        **tkacc_metrics_unfiltered,
        **tkacc_metrics_filtered,

    }])

    # display non-FPR metrics
    display(HTML(metric_table.to_html(index=False)))

    # construct final dictionary for metrics (without stats)
    metric_dict = {
        **cer_metric_unfiltered,
        **cer_metric_filtered,
        **tkacc_metrics_unfiltered,
        **tkacc_metrics_filtered,
        'test_sample_size': len(label_str)
    }

    return metric_dict

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Model Testing

## Load Dataset

In [ ]:
train_sampling_params = {'correction_strategy': 'normal-multiple',
              'correction_distrib': (0, 5),
              'cursor_strategy': 'normal',
              'cursor_bias_type': 'point',
              'cursor_relax': 10,
              'cursor_rep': 'mask',
              'postprocessor': 'none',
              'invert_case_prob': 0.5,
              'log_stuff': False}
tokenizer = T5TokenizerFast.from_pretrained(base_model_name)
if train_sampling_params['cursor_rep'] in ['token']:
    print(f"added {tokenizer.add_special_tokens(custom_special_tokens_dict)} custom special tokens to tokenizer")

inference_sampling_params = dict(train_sampling_params)
inference_sampling_params['inference'] = True
inference_sampling_params['correction_strategy'] = 'normal-multiple'
inference_sampling_params['correction_distrib'] = (0, 5)
inference_sampling_params['cursor_strategy'] = 'uniform'
inference_sampling_params['cursor_rep'] = train_sampling_params['cursor_rep'] if train_sampling_params['cursor_rep'] is not None else 'token'
inference_sampling_params['cursor_relax'] = 10
inference_sampling_params['postprocessor'] = 'unbounded-triplets'
inference_sampling_params['invert_case_prob'] = 0.5
inference_sampling_params['log_stuff'] = False

inference_sampling_strategy = EditSamplingStrategy(**inference_sampling_params)

original_dataset_path = os.path.join(data_dir, 'test_data.csv')
dataset_path = os.path.join(data_dir, 'triplet_test_data_relax10_correction_dev5_stats.csv')
regen = False
if not os.path.exists(dataset_path) or regen:
    test_dataset_gen = TripletCorrectionDatasetWithEditsGenerator(
        data_path=original_dataset_path,
        tokenizer=tokenizer,
        sampling_strategy=inference_sampling_strategy,
    )
    test_dataset_gen.generate_dataset(
        min_classcounts={
            'l': 1500,
            'm': 2500,
            'r': 1500
        },
        dpath=dataset_path
    )

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Load Model

In [ ]:
sweep_name = 'text correction (unbounded triplets cursor strategies)'
model_name = 'correction_normal_multiple_point_uniform_mask_cursor-relax10'
model_path = os.path.join(models_dir, sweep_name, model_name)

model = CursorT5ForConditionalGeneration.from_pretrained(model_path, 'point')

## Test Model

In [ ]:
triplet_runs = [
    ['r'],
    ['l'],
    ['m']
]
mismatch_file=''
triplet_comb = ''
for triplet_classes in triplet_runs:
    triplet_comb = ''.join(triplet_classes)
    print(f'triplet classes: {triplet_classes}')
    # mismatch_file = f'{model_path}/mismatched_cases_triplets-{triplet_comb}-relax10.csv'
    test_dataset = TripletCorrectionDatasetWithEdits(
        data_path=dataset_path,
        tokenizer=tokenizer,
        triplet_classes=triplet_classes,
        scale=PORTION
    )

    # set training arguments
    training_args = Seq2SeqTrainingArguments(
        report_to=[],

        output_dir=''.join([model_path, '/checkpoints/']),

        num_train_epochs=1,

        learning_rate=0,
        weight_decay=0,

        per_device_train_batch_size=0,
        per_device_eval_batch_size=24,

        gradient_accumulation_steps=0,

        optim='adafactor',

        logging_strategy='steps',
        evaluation_strategy='steps',
        save_strategy='steps',

        logging_steps=0.01,
        eval_steps=0.5,
        save_steps=0.5,

        load_best_model_at_end=True,
        metric_for_best_model='loss',
        greater_is_better=False,
        save_total_limit=NUM_SAVES,

        predict_with_generate=True,
        prediction_loss_only=True, # disable using .generate() for evaluation steps during training because it is very slow

        no_cuda=False
    )

    # define training loop
    trainer = EarlyStoppingSeq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=test_dataset,
        eval_dataset=test_dataset,
        compute_metrics=lambda gen_preds : {},
        gen_kwargs_override={**gen_kwargs_override, 'decoder_start_token_id': 0},
        log_stuff=False
    )

    # check vocab
    print(f'model vocab size: {model.config.vocab_size}')
    print(f'tokenizer vocab: {len(tokenizer)}')

    trainer.compute_metrics = compute_metrics
    trainer.args.predict_with_generate = True
    trainer.args.include_inputs_for_metrics = True
    trainer.args.prediction_loss_only = False
    trainer.args.per_device_eval_batch_size = 4

    # # test
    test_evaluation = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix='test')
    print(f"test evaluation: \n{test_evaluation}\n")

    # save test results
    test_evaluation_df = pd.DataFrame(data=[test_evaluation], columns=test_evaluation.keys())
    test_evaluation_df.to_csv(f'{model_path}/test_results triplets-{triplet_comb}-relax10_stats.csv', index=False)


triplet classes: ['r']
preparing data from ./drive/MyDrive/Colab Notebooks/vr-txt-corr/corr_datasets_experimental/DELETION-INSERTION-MULTIPLE-REPLACEMENT-SINGLE/triplet_test_data_relax10_correction_dev5_stats.csv ... 
	reading data ... filtering triplet classes...original len: 25246
updated len: 1500
updated len after dropping nan: 1500
sampling ... 
done


/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:444: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False)
  warnings.warn(


model vocab size: 32100
tokenizer vocab: 32100


[('That must be a lot of kilometres. With the GPS reading still fresh in our minds, we forged ahead, fueled by a mixture of excitement and apprehension. The mountains now seemed closer, their rugged outlines beckoning us forward.', 'lot in kilometers.', '[((5, 7), (5, 7))]', '[((5, 7), (5, 7))]'), ('Do we need to do anything. The question hung in the air, prompting a ripple of reactions. Some nodded in agreement, while others continued to fidget, avoiding eye contact.', 'To do anything here.', '[((6, 6), (6, 7))]', '[((6, 6), (6, 7))]'), ("We will keep teying. Our perseverance will light the way, illuminating pathways we hadn't seen before. Each day brings fresh opportunities to rise and confront our fears head-on.", 'We will keep trying.', '[((3, 4), (3, 4))]', '[((3, 4), (3, 4))]'), ('he extended Freudian theory into adolescence and adulthood l. His research highlighted the significance of peer relationships and their influence on identity formation during teenage years. By integrati

accuracy                 matches           total       \
metric                      top1    top3    top5    top1 top3 top5  top1 top3   
category         subcat                                                         
edit_count       1        0.9379  0.9527  0.9527     317  322  322   338  338   
                 2        0.9577  0.9718  0.9718     136  138  138   142  142   
                 3        0.9444  0.9444  0.9444      17   17   17    18   18   
                 4        1.0000  1.0000  1.0000       2    2    2     2    2   
extra_word_added 0        0.8906  0.9375  0.9375      57   60   60    64   64   
                 1        0.9780  0.9890  0.9890      89   90   90    91   91   
                 2        0.9615  0.9744  0.9744      75   76   76    78   78   
                 3        0.9756  0.9756  0.9756      80   80   80    82   82   
                 4        0.9107  0.9107  0.9107      51   51   51    56   56   
                 5        0.9565  0.9565  0.9565      44   44   44    46   46   
                 6        0.8824  0.9412  0.9412      30   32   32    34   34   
                 7        0.9565  0.9565  0.9565      22   22   22    23   23   
                 8        0.8462  0.8462  0.8462      11   11   11    13   13   
                 9        1.0000  1.0000  1.0000       6    6    6     6    6   
                 10       1.0000  1.0000  1.0000       2    2    2     2    2   
                 11       1.0000  1.0000  1.0000       3    3    3     3    3   
                 12       1.0000  1.0000  1.0000       1    1    1     1    1   
                 13       1.0000  1.0000  1.0000       1    1    1     1    1   
extra_word_count 0        0.9189  0.9730  0.9730      34   36   36    37   37   
                 1        0.9625  0.9750  0.9750      77   78   78    80   80   
                 2        0.9518  0.9759  0.9759      79   81   81    83   83   
                 3        0.9778  0.9778  0.9778      88   88   88    90   90   
                 4        0.8772  0.8772  0.8772      50   50   50    57   57   
                 5        0.9608  0.9608  0.9608      49   49   49    51   51   
                 6        0.9111  0.9556  0.9556      41   43   43    45   45   
                 7        0.9583  0.9583  0.9583      23   23   23    24   24   
                 8        0.8889  0.8889  0.8889      16   16   16    18   18   
                 9        1.0000  1.0000  1.0000       8    8    8     8    8   
                 10       1.0000  1.0000  1.0000       2    2    2     2    2   
                 11       1.0000  1.0000  1.0000       3    3    3     3    3   
                 12       1.0000  1.0000  1.0000       1    1    1     1    1   
                 13       1.0000  1.0000  1.0000       1    1    1     1    1   
part_word_count  1        0.8929  0.9643  0.9643      25   27   27    28   28   
                 2        0.9861  1.0000  1.0000      71   72   72    72   72   
                 3        0.9365  0.9524  0.9524      59   60   60    63   63   
                 4        0.9574  0.9681  0.9681      90   91   91    94   94   
                 5        0.9375  0.9375  0.9375      60   60   60    64   64   
                 6        0.9464  0.9464  0.9464      53   53   53    56   56   
                 7        0.8936  0.9362  0.9362      42   44   44    47   47   
                 8        0.9259  0.9259  0.9259      25   25   25    27   27   
                 9        0.9630  0.9630  0.9630      26   26   26    27   27   
                 10       1.0000  1.0000  1.0000       8    8    8     8    8   
                 11       0.8889  0.8889  0.8889       8    8    8     9    9   
                 12       1.0000  1.0000  1.0000       3    3    3     3    3   
                 13       1.0000  1.0000  1.0000       1    1    1     1    1   
                 14       1.0000  1.0000  1.0000       1    1    1     1    1   

                              
metric                 

...

accuracy                 matches                total  \
metric                      top1    top3    top5    top1   top3   top5   top1   
category         subcat                                                         
edit_count        1       0.9819  0.9929  0.9935   13048  13194  13202  13289   
                  2       0.9720  0.9914  0.9929    5448   5557   5565   5605   
                  3       0.9672  0.9877  0.9877     943    963    963    975   
                  4       0.9412  0.9664  0.9748     112    115    116    119   
                  5       0.9091  1.0000  1.0000      10     11     11     11   
                  6       1.0000  1.0000  1.0000       1      1      1      1   
extra_word_added -1       1.0000  1.0000  1.0000       3      3      3      3   
                  0       0.9289  0.9674  0.9743     941    980    987   1013   
                  1       0.9700  0.9909  0.9909    2137   2183   2183   2203   
                  2       0.9710  0.9900  0.9910    2817   2872   2875   2901   
                  3       0.9821  0.9932  0.9935    3176   3212   3213   3234   
                  4       0.9833  0.9943  0.9950    2950   2983   2985   3000   
                  5       0.9849  0.9944  0.9948    2480   2504   2505   2518   
                  6       0.9821  0.9942  0.9956    2028   2053   2056   2065   
                  7       0.9874  0.9948  0.9948    1337   1347   1347   1354   
                  8       0.9857  0.9967  0.9967     897    907    907    910   
                  9       0.9945  0.9972  0.9972     359    360    360    361   
                  10      0.9961  0.9961  0.9961     258    258    258    259   
                  11      1.0000  1.0000  1.0000      95     95     95     95   
                  12      1.0000  1.0000  1.0000      61     61     61     61   
                  13      1.0000  1.0000  1.0000      16     16     16     16   
                  14      1.0000  1.0000  1.0000       5      5      5      5   
                  15      1.0000  1.0000  1.0000       2      2      2      2   
extra_word_count  0       0.9231  0.9568  0.9700     492    510    517    533   
                  1       0.9673  0.9914  0.9914    1567   1606   1606   1620   
                  2       0.9691  0.9893  0.9897    2444   2495   2496   2522   
                  3       0.9795  0.9921  0.9924    3102   3142   3143   3167   
                  4       0.9834  0.9953  0.9959    3139   3177   3179   3192   
                  5       0.9813  0.9926  0.9933    2780   2812   2814   2833   
                  6       0.9797  0.9930  0.9946    2367   2399   2403   2416   
                  7       0.9851  0.9958  0.9958    1651   1669   1669   1676   
                  8       0.9858  0.9953  0.9953    1044   1054   1054   1059   
                  9       0.9913  0.9935  0.9935     457    458    458    461   
                  10      0.9936  0.9936  0.9936     310    310    310    312   
                  11      1.0000  1.0000  1.0000     108    108    108    108   
                  12      1.0000  1.0000  1.0000      69     69     69     69   
                  13      1.0000  1.0000  1.0000      24     24     24     24   
                  14      1.0000  1.0000  1.0000       6      6      6      6   
                  15      1.0000  1.0000  1.0000       2      2      2      2   
part_word_count   1       0.9124  0.9513  0.9684     375    391    398    411   
                  2       0.9732  0.9905  0.9905    1235   1257   1257   1269   
                  3       0.9768  0.9901  0.9905    2064   2092   2093   2113   
                  4       0.9714  0.9914  0.9914    2921   2981   2981   3007   
                  5       0.9806  0.9936  0.9943    2930   2969   2971   2988   
                  6       0.9840  0.9948  0.9948    2836   2867   2867   2882   
                  7       0.9769  0.9912  0.9935    2539   2576   2582   2599   
                  8       0.9836  0.9947  0.9952    186

Saved detailed statistics to ./drive/MyDrive/Colab Notebooks/vr-txt-corr/saved_models/text correction (unbounded triplets cursor strategies)/correction_normal_multiple_point_uniform_mask_cursor-relax10/stats_triplets-m-relax10.csv

Final Statistics:

Statistics after processing 20880 samples:


accuracy                 matches                total  \
metric                      top1    top3    top5    top1   top3   top5   top1   
category         subcat                                                         
edit_count        1       0.9819  0.9929  0.9935   13114  13261  13269  13356   
                  2       0.9722  0.9915  0.9929    5484   5593   5601   5641   
                  3       0.9675  0.9878  0.9878     953    973    973    985   
                  4       0.9412  0.9664  0.9748     112    115    116    119   
                  5       0.9167  1.0000  1.0000      11     12     12     12   
                  6       1.0000  1.0000  1.0000       1      1      1      1   
extra_word_added -1       1.0000  1.0000  1.0000       3      3      3      3   
                  0       0.9282  0.9676  0.9744     944    984    991   1017   
                  1       0.9702  0.9910  0.9910    2149   2195   2195   2215   
                  2       0.9713  0.9901  0.9911    2840   2895   2898   2924   
                  3       0.9821  0.9932  0.9935    3190   3226   3227   3248   
                  4       0.9834  0.9944  0.9950    2962   2995   2997   3012   
                  5       0.9850  0.9945  0.9949    2501   2525   2526   2539   
                  6       0.9822  0.9942  0.9957    2036   2061   2064   2073   
                  7       0.9875  0.9949  0.9949    1345   1355   1355   1362   
                  8       0.9858  0.9967  0.9967     903    913    913    916   
                  9       0.9946  0.9973  0.9973     365    366    366    367   
                  10      0.9961  0.9961  0.9961     258    258    258    259   
                  11      1.0000  1.0000  1.0000      95     95     95     95   
                  12      1.0000  1.0000  1.0000      61     61     61     61   
                  13      1.0000  1.0000  1.0000      16     16     16     16   
                  14      1.0000  1.0000  1.0000       5      5      5      5   
                  15      1.0000  1.0000  1.0000       2      2      2      2   
extra_word_count  0       0.9213  0.9569  0.9700     492    511    518    534   
                  1       0.9674  0.9914  0.9914    1573   1612   1612   1626   
                  2       0.9692  0.9894  0.9897    2458   2509   2510   2536   
                  3       0.9796  0.9922  0.9925    3120   3160   3161   3185   
                  4       0.9835  0.9953  0.9959    3154   3192   3194   3207   
                  5       0.9814  0.9926  0.9933    2804   2836   2838   2857   
                  6       0.9798  0.9930  0.9946    2377   2409   2413   2426   
                  7       0.9852  0.9958  0.9958    1661   1679   1679   1686   
                  8       0.9860  0.9953  0.9953    1054   1064   1064   1069   
                  9       0.9914  0.9936  0.9936     463    464    464    467   
                  10      0.9936  0.9936  0.9936     310    310    310    312   
                  11      1.0000  1.0000  1.0000     108    108    108    108   
                  12      1.0000  1.0000  1.0000      69     69     69     69   
                  13      1.0000  1.0000  1.0000      24     24     24     24   
                  14      1.0000  1.0000  1.0000       6      6      6      6   
                  15      1.0000  1.0000  1.0000       2      2      2      2   
part_word_count   1       0.9102  0.9515  0.9684     375    392    399    412   
                  2       0.9733  0.9906  0.9906    1241   1263   1263   1275   
                  3       0.9769  0.9901  0.9906    2076   2104   2105   2125   
                  4       0.9715  0.9914  0.9914    2931   2991   2991   3017   
                  5       0.9807  0.9937  0.9943    2947   2986   2988   3005   
                  6       0.9842  0.9948  0.9948    2861   2892   2892   2907   
                  7       0.9770  0.9912  0.9935    2550   2587   2593   2610   
                  8       0.9837  0.9947  0.9953    187

cer_unfiltered,cer_filtered,tkacc_unfiltered/t1acc,tkacc_unfiltered/t3acc,tkacc_unfiltered/t5acc,tkacc_filtered/t1acc,tkacc_filtered/t3acc,tkacc_filtered/t5acc
0.000317,0.000261,0.97196,0.988863,0.991001,0.978174,0.992095,0.99294


test evaluation: 
{'test_sample_size': 20114, 'test_loss': 0.0031918638851493597, 'test_cer_unfiltered': 0.00031731556194199704, 'test_cer_filtered': 0.0002612048833059122, 'test_tkacc_unfiltered/t1acc': 0.9719598289748433, 'test_tkacc_unfiltered/t3acc': 0.9888634781744059, 'test_tkacc_unfiltered/t5acc': 0.9910012926319977, 'test_tkacc_filtered/t1acc': 0.9781744058864472, 'test_tkacc_filtered/t3acc': 0.9920950581684399, 'test_tkacc_filtered/t5acc': 0.992940240628418, 'test_runtime': 23566.3545, 'test_samples_per_second': 0.886, 'test_steps_per_second': 0.222, 'eval_bloss': None}

